# Open Meteo API Tests 

## 1. Imports

In [15]:
import openmeteo_requests
import requests
import pandas as pd

## 2. Set up Client

In [16]:
openmeteo = openmeteo_requests.Client()

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"minutely_15": ["wind_speed_10m", "wind_direction_10m", "wind_gusts_10m", "precipitation"],
	"latitude": 52.52,
	"longitude": 13.41,
    "hourly": None,
	"forecast_days": 1,
}

## 3. Send request

In [17]:
responses = openmeteo.weather_api(url, params = params)
# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Response: {responses[0]}")
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

Response: <openmeteo_sdk.WeatherApiResponse.WeatherApiResponse object at 0x109562140>
Coordinates: 52.52000045776367°N 13.419998168945312°E
Elevation: 38.0 m asl
Timezone difference to GMT+0: 0s


In [18]:
len(responses)


1

## 4. Parse Data into Pandas Dataframe

In [19]:
# Process minutely_15 data. The order of variables needs to be the same as requested.
minutely_15 = response.Minutely15()
minutely_15_wind_speed_10m = minutely_15.Variables(0).ValuesAsNumpy()
minutely_15_wind_direction_10m = minutely_15.Variables(1).ValuesAsNumpy()
minutely_15_wind_gusts_10m = minutely_15.Variables(2).ValuesAsNumpy()
minutely_15_precipitation = minutely_15.Variables(3).ValuesAsNumpy()

minutely_15_data = {"date": pd.date_range(
	start = pd.to_datetime(minutely_15.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(minutely_15.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = minutely_15.Interval()),
	inclusive = "left"
)}

minutely_15_data["wind_speed_10m"] = minutely_15_wind_speed_10m
minutely_15_data["wind_direction_10m"] = minutely_15_wind_direction_10m
minutely_15_data["wind_gusts_10m"] = minutely_15_wind_gusts_10m
minutely_15_data["precipitation"] = minutely_15_precipitation

minutely_15_dataframe = pd.DataFrame(data = minutely_15_data)

minutely_15_dataframe.head(10)

,date,wind_speed_10m,wind_direction_10m,wind_gusts_10m,precipitation
0,2026-04-01 00:00:00+00:00,4.510787,298.610352,13.320000,0.0
1,2026-04-01 00:15:00+00:00,4.510787,298.610352,12.959999,0.0
2,2026-04-01 00:30:00+00:00,4.510787,298.610352,12.599999,0.0
3,2026-04-01 00:45:00+00:00,4.510787,298.610352,11.879999,0.0
4,2026-04-01 01:00:00+00:00,4.510787,298.610352,11.520000,0.0
5,2026-04-01 01:15:00+00:00,4.198285,300.963684,10.799999,0.0
6,2026-04-01 01:30:00+00:00,3.893995,303.690094,9.720000,0.0
7,2026-04-01 01:45:00+00:00,3.600000,306.869965,9.000000,0.0
8,2026-04-01 02:00:00+00:00,3.600000,306.869965,8.640000,0.0
9,2026-04-01 02:15:00+00:00,3.600000,306.869965,9.000000,0.0
